# 🩺 Diabetic Retinopathy Grading — Production Pipeline v19
## 30-Step Complete Pipeline | QWK ≥ 0.90 Target | Colab · Kaggle · Windows · Linux

---
| Step | Description |
|------|-------------|
| 1 | Setup — GPU/CPU/Storage check |
| 2 | Install Requirements |
| 3 | Kaggle Auth & Dataset Download |
| 4 | Load Dataset |
| 5 | Data Cleaning (Laplacian + intensity) |
| 6 | EDA |
| 7 | Label / Imbalance Analysis |
| 8 | Preprocessing Pipeline + Cache |
| 9 | Train/Test Split |
| 10 | K-Fold Splits |
| 11 | Augmentation + Dataset + DataLoader |
| 12 | Model Architecture |
| 13 | Loss + Optimizer + Scheduler |
| 14 | Checkpoint & Resume System |
| 15 | 5-Fold Cross-Validation Training |
| 16 | OOF Predictions |
| 17 | Threshold Optimization |
| 18 | TTA Inference |
| 19 | Final Hold-Out Evaluation |
| 20 | Metrics & Confusion Matrix |
| 21 | Model Export |
| 22 | Grad-CAM++ Explainability |
| 23 | Streamlit Deployment App |
| 24 | Final Summary |


## ✅ Step 1 — Setup: GPU / CPU / Storage Check

In [ ]:
import sys, os, shutil, platform, subprocess
from pathlib import Path

print("="*60)
print("  SYSTEM DIAGNOSTICS")
print("="*60)
print(f"  Python      : {sys.version.split()[0]}")
print(f"  Platform    : {platform.system()} {platform.machine()}")

# GPU check
try:
    import torch
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}       : {p.name}  {p.total_memory/1e9:.1f} GB VRAM")
        print(f"  CUDA        : {torch.version.cuda}")
    elif hasattr(torch.backends,'mps') and torch.backends.mps.is_available():
        print("  Device      : Apple MPS")
    else:
        print("  Device      : CPU only (training will be slow)")
except ImportError:
    print("  PyTorch     : not yet installed")

# Storage check
total, used, free = shutil.disk_usage(Path.home())
print(f"  Disk Free   : {free/1e9:.1f} GB  (need ≥ 5 GB)")
if free < 5e9:
    print("  ⚠️  WARNING: Low disk space!")

# RAM check
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"  RAM         : {ram.total/1e9:.1f} GB total, {ram.available/1e9:.1f} GB free")
except ImportError:
    pass
print("="*60)
print("✅ Setup check complete.")


## 📦 Step 2 — Install Requirements

In [ ]:
import sys, subprocess

def _pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stderr

PACKAGES = [
    "timm>=1.0.3",
    "albumentations>=1.4.0,<2.0.0",
    "opencv-python-headless",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
    "scipy",
    "pyarrow",
    "fastparquet",
    "kaggle",
    "psutil",
    "streamlit",
]

print("Installing packages ...")
failed = []
for pkg in PACKAGES:
    print(f"  {pkg} ...", end=" ", flush=True)
    rc, err = _pip("--upgrade", pkg)
    print("✅" if rc == 0 else "❌")
    if rc != 0:
        print(err[-300:]); failed.append(pkg)

# packaging — Anaconda ships it without RECORD; always force-reinstall
print("  packaging ...", end=" ", flush=True)
rc, err = _pip("--force-reinstall", "--no-deps", "packaging")
print("✅" if rc == 0 else f"❌ {err[-200:]}")
if rc != 0: failed.append("packaging")

# pytorch-grad-cam — try PyPI, fall back to GitHub
print("  pytorch-grad-cam ...", end=" ", flush=True)
rc, _ = _pip("--upgrade", "pytorch-grad-cam")
if rc != 0:
    rc, err2 = _pip("git+https://github.com/jacobgil/pytorch-grad-cam.git")
    if rc != 0:
        print("⚠️  skipped (Grad-CAM cell will be graceful)")
    else:
        print("✅ (GitHub)")
else:
    print("✅")

if failed:
    raise RuntimeError(f"Critical installs failed: {failed}")
print("\n✅ All packages ready.")


## ⚙️ Step 3 — Imports, Global Config & Kaggle Auth

In [ ]:
import os, sys, io, json, gc, time, random, shutil, warnings, zipfile
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from tqdm.auto import tqdm
from packaging.version import Version

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
)
from scipy.optimize import minimize

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  GLOBAL CONFIG
# ─────────────────────────────────────────────────────────────
CFG = {
    "seed"         : 42,
    "model_name"   : "tf_efficientnetv2_b1",
    "n_folds"      : 5,
    "test_size"    : 0.10,     # 10 % hold-out test set
    "lr"           : 3e-4,
    "min_lr"       : 1e-6,
    "weight_decay" : 1e-4,
    "patience"     : 5,
    "min_delta"    : 0.001,
    "grad_clip"    : 1.0,
    "label_smooth" : 0.05,
    "dropout"      : 0.50,
    "phases": [
        {"id":1, "size":224, "batch_size":32, "epochs":15, "freeze":True},
        {"id":2, "size":384, "batch_size":16, "epochs":40, "freeze":False},
        {"id":3, "size":512, "batch_size": 8, "epochs":25, "freeze":False},
    ],
    # Data-cleaning thresholds
    "blur_threshold"      : 50.0,   # Laplacian variance — below = blurry
    "dark_threshold"      : 15.0,   # mean brightness — below = too dark
    "min_nonblack_ratio"  : 0.10,   # fraction of non-black pixels
}

def seed_everything(seed=CFG["seed"]):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False
seed_everything()

# ─────────────────────────────────────────────────────────────
#  DEVICE
# ─────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"🔥 GPU : {torch.cuda.get_device_name(0)}  "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
elif hasattr(torch.backends,"mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps"); print("🍎 Apple MPS")
else:
    DEVICE = torch.device("cpu");  print("💻 CPU only")
USE_AMP = (DEVICE.type == "cuda")

# ─────────────────────────────────────────────────────────────
#  PATHS (auto-detects Kaggle / Colab / Local)
# ─────────────────────────────────────────────────────────────
IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    DATA_DIR     = Path("/kaggle/input/aptos2019-blindness-detection")
    ARTIFACT_DIR = Path("/kaggle/working/dr_artifacts_v19")
elif IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception: pass
    DATA_DIR     = Path("/content/drive/MyDrive/DR_data/aptos2019")
    ARTIFACT_DIR = Path("/content/drive/MyDrive/DR_data/artifacts_v19")
else:
    DATA_DIR     = Path(os.environ.get("DR_DATA",     str(Path.home()/"DR_data"/"aptos2019")))
    ARTIFACT_DIR = Path(os.environ.get("DR_ARTIFACTS",str(Path.home()/"DR_data"/"artifacts_v19")))

IMG_DIR      = DATA_DIR / "train_images"
CSV_PATH     = DATA_DIR / "train.csv"
CACHE_DIR    = ARTIFACT_DIR / "cache"
PLOT_DIR     = ARTIFACT_DIR / "plots"
EXPORT_DIR   = ARTIFACT_DIR / "export"

for d in [ARTIFACT_DIR, CACHE_DIR, PLOT_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

NUM_CLASSES  = 5
GRADE_MAP    = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}
GRADE_COLORS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMAGENET_MEAN = [0.485,0.456,0.406]
IMAGENET_STD  = [0.229,0.224,0.225]

# ─────────────────────────────────────────────────────────────
#  STATE helpers (persist metrics across cells)
# ─────────────────────────────────────────────────────────────
_STATE_FILE = ARTIFACT_DIR / "state_v19.json"
def st_load(): return json.loads(_STATE_FILE.read_text()) if _STATE_FILE.exists() else {}
def st_save(k,v): s=st_load(); s[k]=v; _STATE_FILE.write_text(json.dumps(s,indent=2))

def safe_load(path, map_location="cpu"):
    try:    return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError: return torch.load(path, map_location=map_location)

# ─────────────────────────────────────────────────────────────
#  KAGGLE AUTH (local / Colab only; Kaggle notebooks skip)
# ─────────────────────────────────────────────────────────────
if not IN_KAGGLE:
    kaggle_cfg = Path.home()/".kaggle"/"kaggle.json"
    env_ok = os.environ.get("KAGGLE_KEY") and os.environ.get("KAGGLE_USERNAME")
    if not kaggle_cfg.exists() and not env_ok:
        if IN_COLAB:
            from google.colab import files as _cf
            print("📤 Upload kaggle.json")
            _up = _cf.upload()
            kaggle_cfg.parent.mkdir(parents=True,exist_ok=True)
            for _fn,_d in _up.items(): kaggle_cfg.write_bytes(_d)
        else:
            print("⚠️  Place kaggle.json in ~/.kaggle/ or set KAGGLE_USERNAME + KAGGLE_KEY")
    if kaggle_cfg.exists():
        try: kaggle_cfg.chmod(0o600)
        except Exception: pass
    print("✅ Kaggle auth ready" if kaggle_cfg.exists() or env_ok else "⚠️  Kaggle auth missing")

print(f"\n✅ Config loaded | PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"   DATA_DIR     : {DATA_DIR}")
print(f"   ARTIFACT_DIR : {ARTIFACT_DIR}")


## 📥 Step 4 — Dataset Download & Extraction (APTOS 2019)

In [ ]:
COMPETITION = "aptos2019-blindness-detection"

def _dataset_ok():
    if not CSV_PATH.exists(): return False
    return len(list(IMG_DIR.glob("*.png"))) >= 3000

if _dataset_ok():
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ Dataset present: {n:,} images")

elif IN_KAGGLE:
    src = Path(f"/kaggle/input/{COMPETITION}")
    if src.exists() and not DATA_DIR.exists():
        DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
        DATA_DIR.symlink_to(src)
    print(f"✅ Kaggle input linked: {src}")

else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _zip = DATA_DIR / f"{COMPETITION}.zip"
    if not _zip.exists():
        print("⬇️  Downloading APTOS 2019 (~1.5 GB) ...")
        r = subprocess.run(
            [sys.executable,"-m","kaggle","competitions","download",
             "-c",COMPETITION,"-p",str(DATA_DIR)],
            capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"Download failed:\n{r.stderr}")
        print("✅ Download done.")

    print("📦 Unzipping ...")
    with zipfile.ZipFile(_zip,"r") as z: z.extractall(DATA_DIR)
    for nz in DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(nz,"r") as z: z.extractall(DATA_DIR)
        nz.unlink()
    _zip.unlink(missing_ok=True)
    print(f"✅ {len(list(IMG_DIR.glob('*.png'))):,} images extracted.")

# Verify
if not _dataset_ok():
    raise RuntimeError("Dataset incomplete — check errors above.")
print("✅ Dataset ready.")


## 📂 Step 5 — Load Dataset

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
df_raw["path"]  = df_raw["id_code"].apply(lambda x: str(IMG_DIR/f"{x}.png"))
df_raw["label"] = df_raw["diagnosis"].map(GRADE_MAP)

# Drop rows with missing images
exists_mask = df_raw["path"].apply(lambda p: Path(p).exists())
n_missing = (~exists_mask).sum()
if n_missing: print(f"⚠️  {n_missing} missing images dropped.")
df_raw = df_raw[exists_mask].reset_index(drop=True)

print(f"✅ {len(df_raw):,} images loaded.")
print(df_raw[["diagnosis","label"]].value_counts().sort_index())


## 🧹 Step 6 — Data Cleaning (Laplacian blur + intensity checks)

In [ ]:
def image_quality_flags(path, blur_thr=CFG["blur_threshold"],
                         dark_thr=CFG["dark_threshold"],
                         nonblack_thr=CFG["min_nonblack_ratio"]):
    """
    Returns (is_ok, reason_if_bad).
    Checks: unreadable, blurry (Laplacian), too dark, mostly black borders.
    """
    bgr = cv2.imread(str(path))
    if bgr is None: return False, "unreadable"
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    # Blur check
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    if lap_var < blur_thr: return False, f"blurry(lap={lap_var:.1f})"

    # Darkness check
    mean_br = gray.mean()
    if mean_br < dark_thr: return False, f"dark(mean={mean_br:.1f})"

    # Mostly-black-border check
    nonblack = (gray > 10).mean()
    if nonblack < nonblack_thr: return False, f"black_border(nonblack={nonblack:.2f})"

    return True, ""

_clean_cache = ARTIFACT_DIR / "clean_flags.parquet"
if _clean_cache.exists():
    df_flags = pd.read_parquet(_clean_cache)
    print("✅ [RESUME] Cleaning flags loaded.")
else:
    print("Running image quality checks ...")
    results = [image_quality_flags(p) for p in tqdm(df_raw["path"], leave=False)]
    df_raw["is_ok"]  = [r[0] for r in results]
    df_raw["reason"] = [r[1] for r in results]
    df_flags = df_raw[["id_code","is_ok","reason"]]
    df_flags.to_parquet(_clean_cache, index=False)

df_raw = df_raw.merge(df_flags[["id_code","is_ok","reason"]], on="id_code", how="left") \
           if "is_ok" not in df_raw.columns else df_raw

bad = df_raw[~df_raw["is_ok"]]
print(f"\n  Removed : {len(bad):,} low-quality images")
if len(bad):
    print(bad["reason"].value_counts().to_string())

df = df_raw[df_raw["is_ok"]].reset_index(drop=True)
print(f"  Kept    : {len(df):,} clean images")


## 📊 Step 7 — EDA (Exploratory Data Analysis)

In [ ]:
_eda_file = PLOT_DIR / "eda.png"
if _eda_file.exists():
    print("✅ [RESUME] EDA plot exists.")
    plt.imshow(plt.imread(str(_eda_file))); plt.axis("off"); plt.show()
else:
    counts = [int((df.diagnosis==g).sum()) for g in range(5)]
    labels = [f"G{g}\n{GRADE_MAP[g]}" for g in range(5)]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Bar chart
    bars = axes[0].bar(labels, counts, color=GRADE_COLORS, edgecolor="k", lw=0.6)
    axes[0].set_title("Class Distribution", fontweight="bold")
    for b,n in zip(bars, counts):
        axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+20, str(n), ha="center", fontsize=9)

    # Pie
    axes[1].pie([c/sum(counts)*100 for c in counts], labels=labels,
                colors=GRADE_COLORS, autopct="%1.1f%%", startangle=140)
    axes[1].set_title("Class %", fontweight="bold")

    # Sample grid (5 grades × 3 samples)
    axes[2].axis("off")
    fig2, ax2 = plt.subplots(5, 3, figsize=(9, 15))
    for g in range(5):
        samples = df[df.diagnosis==g].sample(min(3,counts[g]), random_state=42)
        for j, (_, row) in enumerate(samples.iterrows()):
            bgr = cv2.imread(str(row.path))
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr is not None else np.zeros((256,256,3),np.uint8)
            ax2[g][j].imshow(img); ax2[g][j].axis("off")
            if j==0: ax2[g][j].set_ylabel(f"G{g} {GRADE_MAP[g]}", fontsize=8)
    fig2.suptitle("Sample Images per Grade", fontweight="bold")
    fig2.tight_layout()
    fig2.savefig(str(PLOT_DIR/"eda_samples.png"), dpi=100, bbox_inches="tight")
    plt.show(fig2)

    plt.figure(fig.number)
    plt.tight_layout()
    plt.savefig(str(_eda_file), dpi=120, bbox_inches="tight")
    plt.show()
    print(f"✅ EDA saved → {_eda_file}")


## ⚖️ Step 8 — Label / Imbalance Analysis

In [ ]:
counts = df.diagnosis.value_counts().sort_index()
imbalance_ratio = counts.max() / counts.min()
print(f"Imbalance ratio : {imbalance_ratio:.1f}×")
print()

# Class weights for loss
labels_arr = df.diagnosis.values.astype(int)
cls_counts = np.bincount(labels_arr, minlength=NUM_CLASSES).astype(float)
cls_weights_np = len(labels_arr) / (NUM_CLASSES * np.maximum(cls_counts, 1))
cls_weights_np = cls_weights_np / cls_weights_np.sum() * NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(cls_weights_np, dtype=torch.float32)

print("Class weights for loss:")
for g in range(5):
    bar = "█" * int(cls_weights_np[g]*10)
    print(f"  G{g} {GRADE_MAP[g]:15s}: {counts[g]:5d} imgs | weight={cls_weights_np[g]:.3f}  {bar}")

print(f"\n✅ CLASS_WEIGHTS = {np.round(cls_weights_np,3)}")


## 🔬 Step 9 — Preprocessing Pipeline + Cache

In [ ]:
def preprocess_fundus(path_or_img, size=512, sigma_ratio=10, apply_clahe=True):
    """
    STRICT PIPELINE:
    1. Load + BGR→RGB
    2. Auto-crop black borders (mask > 7)
    3. Pad to square, resize to `size`
    4. Circular retinal mask
    5. CLAHE on LAB L-channel
    6. Ben Graham: 4×img − 4×GaussBlur + 128
    Returns uint8 RGB [size×size×3]
    """
    if isinstance(path_or_img, np.ndarray):
        img = path_or_img
    else:
        bgr = cv2.imread(str(path_or_img))
        if bgr is None:
            return np.zeros((size,size,3), np.uint8)
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # 1. Crop black borders
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > 7
    if mask.any():
        rows = np.where(mask.any(1))[0]; cols = np.where(mask.any(0))[0]
        img  = img[rows[0]:rows[-1]+1, cols[0]:cols[-1]+1]

    # 2. Pad to square
    h,w = img.shape[:2]; S = max(h,w)
    ph = (S-h)//2; pb = S-h-ph
    pw = (S-w)//2; pr = S-w-pw
    img = cv2.copyMakeBorder(img, ph,pb,pw,pr, cv2.BORDER_CONSTANT, value=0)

    # 3. Resize
    img = cv2.resize(img, (size,size), interpolation=cv2.INTER_AREA)

    # 4. Circular mask
    cmask = np.zeros((size,size), np.uint8)
    cv2.circle(cmask, (size//2,size//2), int(size//2*0.97), 255, -1)
    img[cmask==0] = 0

    # 5. CLAHE
    if apply_clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        lab[:,:,0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(lab[:,:,0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    # 6. Ben Graham sharpening
    sigma = max((size//sigma_ratio)|1, 1)
    blur  = cv2.GaussianBlur(img,(0,0),sigmaX=sigma)
    img   = cv2.addWeighted(img,4,blur,-4,128)
    img[cmask==0] = 0

    return img

# ── Preprocessing Cache ───────────────────────────────────────
CACHE_SIZE = 224   # cache at smallest phase size to save disk
_cache_flag = CACHE_DIR / "_done.flag"

def get_cached_path(orig_path):
    stem = Path(orig_path).stem
    return CACHE_DIR / f"{stem}.png"

if _cache_flag.exists():
    print(f"✅ [RESUME] Cache exists at {CACHE_DIR}")
else:
    print(f"Building preprocessing cache ({len(df)} images @ {CACHE_SIZE}px) ...")
    for _, row in tqdm(df.iterrows(), total=len(df), leave=False):
        dest = get_cached_path(row.path)
        if not dest.exists():
            proc = preprocess_fundus(row.path, size=CACHE_SIZE)
            cv2.imwrite(str(dest), cv2.cvtColor(proc, cv2.COLOR_RGB2BGR))
    _cache_flag.touch()
    print(f"✅ Cache built → {CACHE_DIR}")

# Latency sanity check
t0 = time.time()
for _ in range(5): preprocess_fundus(df.path.iloc[0], size=512)
print(f"   Preprocess latency: {(time.time()-t0)/5*1e3:.1f} ms/img")


## ✂️ Step 10 — Train / Test Split (Hold-Out)

In [ ]:
_split_file = ARTIFACT_DIR / "train_test_split.parquet"
if _split_file.exists():
    df_split = pd.read_parquet(_split_file)
    df["split"] = df_split["split"].values
    print("✅ [RESUME] Split loaded.")
else:
    # Stratified split: 90% train+val, 10% test
    train_idx, test_idx = train_test_split(
        df.index, test_size=CFG["test_size"],
        stratify=df.diagnosis, random_state=CFG["seed"])
    df["split"] = "train"
    df.loc[test_idx, "split"] = "test"
    df[["id_code","split"]].to_parquet(_split_file, index=False)
    print("✅ Train/test split created.")

df_trainval = df[df.split=="train"].reset_index(drop=True)
df_test     = df[df.split=="test" ].reset_index(drop=True)
print(f"   Train+Val: {len(df_trainval):,}   Test (held-out): {len(df_test):,}")
print(f"   Test grade dist: {dict(df_test.diagnosis.value_counts().sort_index())}")


## 🔀 Step 11 — Stratified K-Fold Splits

In [ ]:
_kfold_file = ARTIFACT_DIR / "kfold_splits.parquet"
if _kfold_file.exists():
    df_folds = pd.read_parquet(_kfold_file)
    df_trainval["fold"] = df_folds["fold"].values
    print("✅ [RESUME] K-Fold splits loaded.")
else:
    skf = StratifiedKFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
    df_trainval["fold"] = -1
    for fi,(_, vi) in enumerate(skf.split(df_trainval, df_trainval.diagnosis)):
        df_trainval.loc[vi,"fold"] = fi
    df_trainval[["id_code","fold"]].to_parquet(_kfold_file, index=False)
    print("✅ 5-Fold splits created.")

print("\nFold distribution:")
for f in range(CFG["n_folds"]):
    n  = (df_trainval.fold==f).sum()
    gd = df_trainval[df_trainval.fold==f].diagnosis.value_counts().sort_index()
    gs = " | ".join(f"G{g}:{c}" for g,c in gd.items())
    print(f"  Fold {f}: {n:5d}  [{gs}]")


## 🔄 Step 12 — Augmentation, Dataset & DataLoader

In [ ]:
_A_NEW = Version(A.__version__) >= Version("1.4.0")

def _gauss_noise():
    return (A.GaussNoise(std_range=(0.03,0.10), p=0.2) if _A_NEW
            else A.GaussNoise(var_limit=(10,40), p=0.2))

def _coarse_drop(sz):
    h = sz//16
    return (A.CoarseDropout(num_holes_range=(1,6),hole_height_range=(h,h),
                            hole_width_range=(h,h),p=0.2) if _A_NEW
            else A.CoarseDropout(max_holes=6,max_height=h,max_width=h,p=0.2))

def get_train_transform(sz):
    return A.Compose([
        A.RandomResizedCrop(height=sz, width=sz, scale=(0.8,1.0), ratio=(0.9,1.1)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05,scale_limit=0.1,rotate_limit=15,p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15,contrast_limit=0.15,p=0.4),
        A.CLAHE(clip_limit=2.0,p=0.3),
        A.HueSaturationValue(hue_shift_limit=10,sat_shift_limit=20,val_shift_limit=10,p=0.3),
        A.RandomGamma(gamma_limit=(80,120),p=0.3),
        _gauss_noise(),
        A.MotionBlur(blur_limit=3,p=0.1),
        _coarse_drop(sz),
        A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_val_transform(sz):
    return A.Compose([
        A.Resize(sz,sz),
        A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
        ToTensorV2(),
    ])

def get_tta_transforms(sz):
    base = [A.Resize(sz,sz), A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD), ToTensorV2()]
    return [
        A.Compose(base),
        A.Compose([A.HorizontalFlip(p=1.0)] + base),
        A.Compose([A.VerticalFlip(p=1.0)]   + base),
        A.Compose([A.RandomBrightnessContrast(brightness_limit=0.1,contrast_limit=0.1,p=1.0)] + base),
        A.Compose([A.Rotate(limit=10,p=1.0)] + base),
    ]

# ── Dataset ───────────────────────────────────────────────────
class DRDataset(Dataset):
    def __init__(self, df, transform=None, img_size=512, use_cache=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size
        self.use_cache = use_cache

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Use cache if available and size matches
        cached = get_cached_path(row.path)
        if self.use_cache and cached.exists() and self.img_size == CACHE_SIZE:
            bgr = cv2.imread(str(cached))
            img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        else:
            img = preprocess_fundus(row.path, size=self.img_size)

        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2,0,1)).float()/255.0

        label = torch.tensor(int(row.diagnosis), dtype=torch.long)
        return img, label

def make_weighted_loader(df_split, dataset, batch_size, drop_last=False):
    labs   = df_split.diagnosis.values.astype(int)
    cnts   = np.bincount(labs, minlength=NUM_CLASSES).astype(float)
    w_cls  = 1.0 / np.maximum(cnts,1)
    s_wts  = torch.tensor([w_cls[l] for l in labs], dtype=torch.float)
    sampler= WeightedRandomSampler(s_wts, len(s_wts), replacement=True)
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=nw, pin_memory=(DEVICE.type=="cuda"),
                      drop_last=drop_last, persistent_workers=(nw>0))

def make_loader(dataset, batch_size, shuffle=False, drop_last=False):
    nw = min(4, os.cpu_count() or 1)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=nw, pin_memory=(DEVICE.type=="cuda"),
                      drop_last=drop_last, persistent_workers=(nw>0))

print(f"✅ Transforms & Dataset classes ready | albumentations {A.__version__}")


## 🏗️ Step 13 — Model Architecture (EfficientNetV2 + Custom Head)

In [ ]:
class DRModel(nn.Module):
    """
    Backbone: tf_efficientnetv2_b1 (ImageNet pretrained)
    Head: GlobalAvgPool → BN → Linear(256,ReLU) → Dropout → Linear(5)
    """
    def __init__(self, model_name=CFG["model_name"], num_classes=NUM_CLASSES,
                 pretrained=True, dropout=CFG["dropout"]):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, num_classes=0, global_pool="avg")
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat),
            nn.Linear(feat, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)

    def unfreeze_top(self, n=4):
        self.freeze_backbone()
        if hasattr(self.backbone,"blocks"):
            for blk in list(self.backbone.blocks)[-n:]:
                for p in blk.parameters(): p.requires_grad_(True)
        for attr in ("conv_head","bn2","norm_head","norm"):
            if hasattr(self.backbone,attr):
                for p in getattr(self.backbone,attr).parameters(): p.requires_grad_(True)

# Sanity check
_m = DRModel(pretrained=False).to(DEVICE)
_x = torch.randn(2,3,224,224).to(DEVICE)
_o = _m(_x)
assert _o.shape == (2, NUM_CLASSES), f"Bad output shape: {_o.shape}"
print(f"✅ DRModel OK: input {list(_x.shape)} → output {list(_o.shape)}")
total_params = sum(p.numel() for p in _m.parameters())
trainable    = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"   Params: {total_params/1e6:.1f}M total, {trainable/1e6:.1f}M trainable")
del _m,_x,_o; gc.collect()


## ⚖️ Step 14 — Loss, Optimizer, Scheduler & Metrics

In [ ]:
# ── Hybrid Loss: 0.5 × WeightedCE + 0.5 × FocalLoss ─────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
        self.weight = weight; self.ls = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.ls, reduction="none")
        pt  = torch.exp(-ce)
        return (self.alpha * (1-pt)**self.gamma * ce).mean()

class HybridLoss(nn.Module):
    """
    0.5 × Weighted CrossEntropy (label_smoothing=0.05)
    + 0.5 × Focal Loss (γ=2, α=0.25)
    """
    def __init__(self, class_weights=None):
        super().__init__()
        w = class_weights
        self.ce    = nn.CrossEntropyLoss(weight=w, label_smoothing=CFG["label_smooth"])
        self.focal = FocalLoss(alpha=0.25, gamma=2.0, weight=w,
                               label_smoothing=CFG["label_smooth"])

    def forward(self, logits, targets):
        return 0.5*self.ce(logits, targets) + 0.5*self.focal(logits, targets)

# ── Metrics ───────────────────────────────────────────────────────
def qwk(y_true, y_pred):
    return cohen_kappa_score(np.array(y_true), np.array(y_pred), weights="quadratic")

# ── Threshold optimizer (Nelder-Mead on OOF softmax) ─────────────
class ThresholdOptimizer:
    """Fits 4 cut-points on softmax probabilities to maximise QWK."""
    def __init__(self):
        self.thresholds_ = np.array([0.5,1.5,2.5,3.5])

    def _loss(self, thresholds, probs, y_true):
        # Convert softmax → scalar via weighted sum, then threshold
        scalar = probs @ np.arange(NUM_CLASSES)
        cuts   = np.sort(thresholds)
        preds  = pd.cut(scalar, bins=[-np.inf]+list(cuts)+[np.inf],
                        labels=list(range(NUM_CLASSES))).astype(int)
        return -cohen_kappa_score(y_true, preds, weights="quadratic")

    def fit(self, probs, y_true):
        res = minimize(self._loss, self.thresholds_, args=(probs, y_true),
                       method="Nelder-Mead",
                       options={"maxiter":2000,"xatol":1e-6,"fatol":1e-9})
        self.thresholds_ = np.sort(res.x)
        return self

    def predict(self, probs):
        scalar = probs @ np.arange(NUM_CLASSES)
        return np.clip(
            pd.cut(scalar, bins=[-np.inf]+list(self.thresholds_)+[np.inf],
                   labels=list(range(NUM_CLASSES))).astype(int), 0, 4)

print("✅ HybridLoss, FocalLoss, ThresholdOptimizer, qwk() defined.")


## 💾 Step 15 — Checkpoint & Resume System

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, scaler,
                    epoch, batch, phase_id, best_qwk, thresholds, history):
    torch.save({
        "model"      : model.state_dict(),
        "optimizer"  : optimizer.state_dict(),
        "scheduler"  : scheduler.state_dict(),
        "scaler"     : scaler.state_dict() if scaler else None,
        "epoch"      : epoch,
        "batch"      : batch,
        "phase_id"   : phase_id,
        "best_qwk"   : best_qwk,
        "thresholds" : thresholds,
        "history"    : history,
    }, path)

def load_checkpoint(path, model, optimizer=None, scheduler=None,
                    scaler=None, map_location="cpu"):
    ckpt = safe_load(path, map_location)
    model.load_state_dict(ckpt["model"])
    if optimizer and ckpt.get("optimizer"):
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler and ckpt.get("scheduler"):
        scheduler.load_state_dict(ckpt["scheduler"])
    if scaler and ckpt.get("scaler"):
        scaler.load_state_dict(ckpt["scaler"])
    return ckpt

print("✅ Checkpoint helpers defined.")


## 🏋️ Step 16 — 5-Fold Cross-Validation Training (Phase-Wise + Early Stopping)

In [ ]:
def run_epoch_train(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc="  train", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast("cuda"):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss / len(loader)

@torch.no_grad()
def run_epoch_val(model, loader, scaler, device):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="  val  ", leave=False):
        imgs = imgs.to(device)
        if scaler:
            with torch.amp.autocast("cuda"): logits = model(imgs)
        else:
            logits = model(imgs)
        all_probs.append(F.softmax(logits, dim=1).cpu().float().numpy())
        all_labels.extend(labels.numpy())
    return np.concatenate(all_probs), np.array(all_labels)

# ─── Main K-Fold loop ──────────────────────────────────────────
fold_best_qwks = []
oof_probs      = np.zeros((len(df_trainval), NUM_CLASSES), dtype=np.float32)
oof_labels_arr = df_trainval.diagnosis.values.copy()
global_thr_opt = ThresholdOptimizer()
scaler_global  = torch.amp.GradScaler("cuda") if USE_AMP else None

print("="*68)
print(f"  5-FOLD CV | {CFG['model_name']} | device={DEVICE}")
print("="*68)

for fold in range(CFG["n_folds"]):
    ckpt_best  = ARTIFACT_DIR / f"fold{fold}_best.pt"
    oof_file   = ARTIFACT_DIR / f"fold{fold}_oof_probs.npy"
    done_flag  = ARTIFACT_DIR / f"_done_fold{fold}.flag"

    if done_flag.exists() and ckpt_best.exists():
        prev = safe_load(ckpt_best, "cpu")
        fold_best_qwks.append(prev.get("best_qwk",0.0))
        if oof_file.exists():
            val_idx = df_trainval[df_trainval.fold==fold].index
            oof_probs[val_idx] = np.load(str(oof_file))
        print(f"  ✅ [RESUME] Fold {fold} | QWK={fold_best_qwks[-1]:.4f}")
        continue

    print(f"\n  {'━'*20}  FOLD {fold}  {'━'*20}")
    df_tr  = df_trainval[df_trainval.fold!=fold].reset_index(drop=True)
    df_va  = df_trainval[df_trainval.fold==fold].reset_index(drop=True)
    val_idx = df_trainval[df_trainval.fold==fold].index

    model     = DRModel(pretrained=True).to(DEVICE)
    criterion = HybridLoss(CLASS_WEIGHTS.to(DEVICE))
    thr_opt   = ThresholdOptimizer()

    fold_best_qwk  = -1.0
    best_state     = None
    best_thresholds= thr_opt.thresholds_.copy()
    history        = []

    for phase in CFG["phases"]:
        pid, sz, bs, n_ep = phase["id"], phase["size"], phase["batch_size"], phase["epochs"]
        print(f"\n  Phase {pid} | {sz}px | {n_ep} epochs | "
              f"{'frozen' if phase['freeze'] else 'unfrozen'}")

        if phase["freeze"]:
            model.freeze_backbone()
        elif pid == 2:
            model.unfreeze_top(n=4)
        else:
            model.unfreeze_backbone()

        tr_ds  = DRDataset(df_tr, get_train_transform(sz), img_size=sz, use_cache=False)
        va_ds  = DRDataset(df_va, get_val_transform(sz),   img_size=sz, use_cache=False)
        tr_ld  = make_weighted_loader(df_tr, tr_ds, bs, drop_last=True)
        va_ld  = make_loader(va_ds, bs)

        lr_phase = CFG["lr"] / (3**(pid-1))
        optimizer= torch.optim.AdamW(
            filter(lambda p:p.requires_grad, model.parameters()),
            lr=lr_phase, weight_decay=CFG["weight_decay"])
        scheduler= torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=n_ep*len(tr_ld),
            eta_min=CFG["min_lr"])

        patience_cnt = 0
        for ep in range(n_ep):
            tr_loss = run_epoch_train(
                model, tr_ld, criterion, optimizer, scheduler, scaler_global, DEVICE)
            val_probs, val_labs = run_epoch_val(model, va_ld, scaler_global, DEVICE)

            # Threshold-optimize on this fold's val preds
            thr_opt.fit(val_probs, val_labs)
            val_preds = thr_opt.predict(val_probs)
            val_qwk   = qwk(val_labs, val_preds)
            val_acc   = accuracy_score(val_labs, val_preds)

            improved = val_qwk > fold_best_qwk + CFG["min_delta"]
            if improved:
                fold_best_qwk   = val_qwk
                best_state      = deepcopy(model.state_dict())
                best_thresholds = thr_opt.thresholds_.copy()
                patience_cnt    = 0
                save_checkpoint(
                    ckpt_best, model, optimizer, scheduler, scaler_global,
                    ep, 0, pid, fold_best_qwk, best_thresholds, history)
            else:
                patience_cnt += 1

            history.append({"fold":fold,"phase":pid,"epoch":ep,
                            "tr_loss":tr_loss,"val_qwk":val_qwk,"val_acc":val_acc})
            star = " ★" if improved else ""
            print(f"    P{pid} Ep{ep+1:02d}/{n_ep}: "
                  f"loss={tr_loss:.4f}  QWK={val_qwk:.4f}  Acc={val_acc*100:.1f}%{star}")

            if patience_cnt >= CFG["patience"]:
                print(f"    ↳ Early stop (patience={CFG['patience']})")
                break

    # OOF probs (best model, 512px, no TTA)
    model.load_state_dict(best_state)
    va_ds_f = DRDataset(df_va, get_val_transform(512), img_size=512, use_cache=False)
    va_ld_f = make_loader(va_ds_f, 8)
    fold_probs, _ = run_epoch_val(model, va_ld_f, scaler_global, DEVICE)

    oof_probs[val_idx] = fold_probs[:len(val_idx)]
    np.save(str(oof_file), fold_probs[:len(val_idx)])
    fold_best_qwks.append(fold_best_qwk)
    done_flag.touch()
    print(f"  ✅ Fold {fold} done — Best QWK: {fold_best_qwk:.4f}")
    del model; gc.collect()
    if DEVICE.type=="cuda": torch.cuda.empty_cache()

np.save(str(ARTIFACT_DIR/"oof_probs.npy"),  oof_probs)
np.save(str(ARTIFACT_DIR/"oof_labels.npy"), oof_labels_arr)

print("\n" + "="*68)
for i,q in enumerate(fold_best_qwks): print(f"  Fold {i}: QWK = {q:.4f}")
print(f"  Mean : {np.mean(fold_best_qwks):.4f} ± {np.std(fold_best_qwks):.4f}")
print("="*68)


## 📐 Step 17 — OOF Evaluation & Threshold Optimization

In [ ]:
oof_probs_l  = np.load(str(ARTIFACT_DIR/"oof_probs.npy"))
oof_labels_l = np.load(str(ARTIFACT_DIR/"oof_labels.npy"))

# Optimize thresholds on full OOF
print("Optimizing thresholds on OOF predictions ...")
global_thr_opt.fit(oof_probs_l, oof_labels_l)
oof_preds_opt = global_thr_opt.predict(oof_probs_l)

# Baseline: argmax
oof_preds_argmax = oof_probs_l.argmax(axis=1)

oof_qwk_opt  = qwk(oof_labels_l, oof_preds_opt)
oof_qwk_base = qwk(oof_labels_l, oof_preds_argmax)
oof_acc_opt  = accuracy_score(oof_labels_l, oof_preds_opt)

print(f"  OOF QWK (argmax)    : {oof_qwk_base:.4f}")
print(f"  OOF QWK (optimized) : {oof_qwk_opt:.4f}  ← use this")
print(f"  OOF Acc (optimized) : {oof_acc_opt*100:.2f}%")
print(f"  Thresholds          : {np.round(global_thr_opt.thresholds_,3)}")

np.save(str(ARTIFACT_DIR/"oof_preds.npy"),   oof_preds_opt)
np.save(str(ARTIFACT_DIR/"thresholds.npy"),  global_thr_opt.thresholds_)
st_save("oof_qwk", float(oof_qwk_opt))
st_save("oof_acc", float(oof_acc_opt))

TARGET_MET = oof_qwk_opt >= 0.90
print(f"\n  {'✅' if TARGET_MET else '⚠️ '} QWK target (≥0.90): {'MET' if TARGET_MET else 'NOT YET MET'}")


## 🔁 Step 18 — TTA Ensemble Inference on Test Set

In [ ]:
@torch.no_grad()
def tta_predict(df_infer, tta_size=512, n_models="all"):
    """
    Runs all fold models × 5-view TTA on df_infer.
    Returns averaged softmax probabilities.
    """
    tta_tfs     = get_tta_transforms(tta_size)
    fold_paths  = sorted(ARTIFACT_DIR.glob("fold*_best.pt"))
    if not fold_paths:
        raise RuntimeError("No fold checkpoints found — run training first.")

    ensemble = np.zeros((len(df_infer), NUM_CLASSES), dtype=np.float32)
    n_contrib = 0

    for ckpt_path in fold_paths:
        ckpt  = safe_load(ckpt_path, DEVICE)
        model = DRModel(pretrained=False).to(DEVICE)
        model.load_state_dict(ckpt["model"]); model.eval()

        fold_sum = np.zeros((len(df_infer), NUM_CLASSES), dtype=np.float32)
        for tf in tta_tfs:
            ds = DRDataset(df_infer, tf, img_size=tta_size, use_cache=False)
            ld = make_loader(ds, 8)
            probs_list = []
            for imgs,_ in tqdm(ld, desc=f"  {ckpt_path.stem} TTA", leave=False):
                imgs = imgs.to(DEVICE)
                if USE_AMP:
                    with torch.amp.autocast("cuda"): logits = model(imgs)
                else:
                    logits = model(imgs)
                probs_list.append(F.softmax(logits,dim=1).cpu().float().numpy())
            fold_sum += np.concatenate(probs_list)[:len(df_infer)]
        ensemble += fold_sum / len(tta_tfs)
        n_contrib += 1
        del model; gc.collect()
        if DEVICE.type=="cuda": torch.cuda.empty_cache()

    return ensemble / n_contrib

print("Running TTA inference on hold-out test set ...")
test_probs = tta_predict(df_test, tta_size=512)

# Load optimized thresholds
_thr = np.load(str(ARTIFACT_DIR/"thresholds.npy"))
global_thr_opt.thresholds_ = _thr
test_preds  = global_thr_opt.predict(test_probs)
test_labels = df_test.diagnosis.values

np.save(str(ARTIFACT_DIR/"test_probs.npy"), test_probs)
np.save(str(ARTIFACT_DIR/"test_preds.npy"), test_preds)
print(f"✅ TTA inference done on {len(df_test)} test samples.")


## 📈 Step 19 — Metrics, Confusion Matrix & Per-Class Report

In [ ]:
test_probs_l  = np.load(str(ARTIFACT_DIR/"test_probs.npy"))
test_preds_l  = np.load(str(ARTIFACT_DIR/"test_preds.npy"))
test_labels_l = df_test.diagnosis.values

test_qwk = qwk(test_labels_l, test_preds_l)
test_acc = accuracy_score(test_labels_l, test_preds_l)
st_save("test_qwk", float(test_qwk))
st_save("test_acc", float(test_acc))

print(f"  Test QWK : {test_qwk:.4f}  {'✅' if test_qwk>=0.90 else '⚠️ '}")
print(f"  Test Acc : {test_acc*100:.2f}%")
print()

# Confusion matrix + per-class metrics
cm  = confusion_matrix(test_labels_l, test_preds_l)
fig, axes = plt.subplots(1,2, figsize=(16,6))

ConfusionMatrixDisplay(cm, display_labels=[f"G{i}" for i in range(5)]).plot(
    ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Test Confusion Matrix\nQWK={test_qwk:.4f}  Acc={test_acc*100:.1f}%",
                  fontweight="bold")

report = classification_report(
    test_labels_l, test_preds_l,
    target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)], output_dict=True)
recalls    = [report[f"G{i} {GRADE_MAP[i]}"]["recall"]    for i in range(5)]
precisions = [report[f"G{i} {GRADE_MAP[i]}"]["precision"] for i in range(5)]
x = np.arange(5); w = 0.35
axes[1].bar(x-w/2, recalls,    width=w, color=GRADE_COLORS, label="Recall",    alpha=0.9)
axes[1].bar(x+w/2, precisions, width=w, color=GRADE_COLORS, label="Precision", alpha=0.5, hatch="//")
axes[1].set_xticks(x); axes[1].set_xticklabels([f"G{i}" for i in range(5)])
axes[1].set_ylim(0,1.15); axes[1].legend(); axes[1].set_title("Per-Class Recall & Precision", fontweight="bold")
plt.tight_layout()
plt.savefig(str(PLOT_DIR/"test_metrics.png"), dpi=120, bbox_inches="tight")
plt.show()

print(classification_report(
    test_labels_l, test_preds_l,
    target_names=[f"G{i} {GRADE_MAP[i]}" for i in range(5)]))


## 💾 Step 20 — Model Export (Best Fold + Thresholds + Label Map)

In [ ]:
# Find best fold
_best_fold = int(np.argmax(fold_best_qwks)) if fold_best_qwks else 0
_src_ckpt  = ARTIFACT_DIR / f"fold{_best_fold}_best.pt"
_dst_model = EXPORT_DIR / "dr_model_final.pt"

# Load best model
_ckpt = safe_load(_src_ckpt, "cpu")
_export_model = DRModel(pretrained=False)
_export_model.load_state_dict(_ckpt["model"])
_export_model.eval()

# Save clean export (model weights only)
torch.save({
    "model_state_dict" : _export_model.state_dict(),
    "model_name"       : CFG["model_name"],
    "num_classes"      : NUM_CLASSES,
    "grade_map"        : GRADE_MAP,
    "thresholds"       : global_thr_opt.thresholds_.tolist(),
    "imagenet_mean"    : IMAGENET_MEAN,
    "imagenet_std"     : IMAGENET_STD,
    "test_qwk"         : float(test_qwk),
    "oof_qwk"          : float(oof_qwk_opt),
}, str(_dst_model))

# Save thresholds JSON
(_EXPORT_DIR / "thresholds.json") if False else None
import json as _json
(_dst_thresh := EXPORT_DIR/"thresholds.json").write_text(
    _json.dumps({"thresholds": global_thr_opt.thresholds_.tolist(),
                 "grade_map" : GRADE_MAP}, indent=2))

# Save label map
(EXPORT_DIR/"label_map.json").write_text(
    _json.dumps(GRADE_MAP, indent=2))

print(f"✅ Model exported → {_dst_model}  ({_dst_model.stat().st_size/1e6:.1f} MB)")
print(f"✅ Thresholds   → {_dst_thresh}")
print(f"   Best fold   : {_best_fold}  (QWK={fold_best_qwks[_best_fold]:.4f})")
del _export_model; gc.collect()


## 🎨 Step 21 — Grad-CAM++ Explainability

In [ ]:
try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image

    _bfi  = int(np.argmax(fold_best_qwks)) if fold_best_qwks else 0
    _ckpt = safe_load(ARTIFACT_DIR/f"fold{_bfi}_best.pt", DEVICE)
    _m    = DRModel(pretrained=False).to(DEVICE)
    _m.load_state_dict(_ckpt["model"]); _m.eval()

    # Robust target-layer detection
    if hasattr(_m.backbone,"blocks"):
        _tgt = [_m.backbone.blocks[-1][-1]]
    else:
        _children = list(_m.backbone.children())
        _tgt = [_children[-1] if isinstance(_children[-1], nn.Module) else _children[-2]]

    cam = GradCAMPlusPlus(model=_m, target_layers=_tgt)

    fig, axes = plt.subplots(2,5, figsize=(22,9))
    for grade in range(5):
        sample = df[df.diagnosis==grade].sample(1, random_state=42).iloc[0]
        raw    = preprocess_fundus(sample.path, size=512)
        tf     = get_val_transform(512)
        tensor = tf(image=raw)["image"].unsqueeze(0).to(DEVICE)

        gc_map = cam(input_tensor=tensor, targets=None)[0]
        vis    = show_cam_on_image(raw.astype(np.float32)/255.0, gc_map, use_rgb=True)

        axes[0][grade].imshow(raw)
        axes[0][grade].set_title(f"G{grade}: {GRADE_MAP[grade]}", fontsize=9)
        axes[0][grade].axis("off")
        axes[1][grade].imshow(vis)
        axes[1][grade].set_title("Grad-CAM++", fontsize=9)
        axes[1][grade].axis("off")

    plt.suptitle("Grad-CAM++ — Model Attention per DR Grade", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(str(PLOT_DIR/"gradcam.png"), dpi=120, bbox_inches="tight")
    plt.show()
    del _m, cam; gc.collect()
    print("✅ Grad-CAM++ complete.")

except ImportError:
    print("⚠️  pytorch-grad-cam not installed — Cell skipped gracefully.")
    print("   To install: pip install pytorch-grad-cam")
except Exception as e:
    print(f"⚠️  Grad-CAM error: {e}")


## 🌐 Step 22 — Streamlit Deployment App

Run the generated file with: `streamlit run dr_app.py`


In [ ]:
APP_CODE = '''
import streamlit as st
import torch, cv2, numpy as np, json
from pathlib import Path
from PIL import Image
import torch.nn as nn
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Config ────────────────────────────────────────────────────────────────────
EXPORT_DIR = Path(__file__).parent / "export"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Model ─────────────────────────────────────────────────────────────────────
@st.cache_resource
def load_model():
    bundle = torch.load(str(EXPORT_DIR/"dr_model_final.pt"), map_location=DEVICE,
                        weights_only=False)
    class DRModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = timm.create_model(
                bundle["model_name"], pretrained=False, num_classes=0, global_pool="avg")
            feat = self.backbone.num_features
            self.head = nn.Sequential(
                nn.BatchNorm1d(feat), nn.Linear(feat,256),
                nn.ReLU(True), nn.Dropout(0.5), nn.Linear(256,5))
        def forward(self, x): return self.head(self.backbone(x))
    m = DRModel().to(DEVICE)
    m.load_state_dict(bundle["model_state_dict"])
    m.eval()
    thresholds = np.array(bundle["thresholds"])
    grade_map  = bundle["grade_map"]
    return m, thresholds, grade_map

# ── Preprocessing ─────────────────────────────────────────────────────────────
def preprocess(img_rgb, size=512):
    img = cv2.resize(img_rgb, (size,size), interpolation=cv2.INTER_AREA)
    cmask = np.zeros((size,size),np.uint8)
    cv2.circle(cmask,(size//2,size//2),int(size//2*0.97),255,-1)
    img[cmask==0] = 0
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    lab[:,:,0] = cv2.createCLAHE(2.0,(8,8)).apply(lab[:,:,0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    blur = cv2.GaussianBlur(img,(0,0),sigmaX=51)
    img  = cv2.addWeighted(img,4,blur,-4,128)
    img[cmask==0] = 0
    return img

def predict(model, thresholds, img_rgb):
    proc = preprocess(img_rgb)
    tf   = A.Compose([A.Resize(512,512),
                      A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
                      ToTensorV2()])
    tensor = tf(image=proc)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(tensor),dim=1).cpu().numpy()[0]
    scalar = probs @ np.arange(5)
    import pandas as pd
    grade  = int(pd.cut([scalar], bins=[-np.inf]+list(np.sort(thresholds))+[np.inf],
                        labels=[0,1,2,3,4]).astype(int)[0])
    return grade, probs

# ── Streamlit UI ──────────────────────────────────────────────────────────────
st.set_page_config(page_title="DR Grading", page_icon="🩺", layout="wide")
st.title("🩺 Diabetic Retinopathy Grading")
st.caption("Upload a fundus image — model returns DR grade 0-4 with confidence")

col1, col2 = st.columns([1,1])
with col1:
    uploaded = st.file_uploader("Upload fundus image", type=["png","jpg","jpeg"])

if uploaded:
    model, thresholds, grade_map = load_model()
    pil_img = Image.open(uploaded).convert("RGB")
    img_rgb = np.array(pil_img)

    with col1:
        st.image(pil_img, caption="Input image", use_container_width=True)

    with st.spinner("Running inference ..."):
        grade, probs = predict(model, thresholds, img_rgb)

    with col2:
        GRADE_MAP = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}
        COLORS    = ["green","yellow","orange","red","purple"]
        st.metric("Predicted Grade", f"Grade {grade}: {GRADE_MAP[grade]}")
        st.progress(int(probs[grade]*100))
        st.subheader("Confidence per Grade")
        import pandas as pd
        chart_df = pd.DataFrame({"Grade":[f"G{i} {GRADE_MAP[i]}" for i in range(5)],
                                  "Probability":probs})
        st.bar_chart(chart_df.set_index("Grade"))
        if grade >= 2:
            st.warning("⚠️  Moderate-Severe DR detected — refer to ophthalmologist.")
        else:
            st.success("✅ No or mild DR detected.")

st.markdown("---")
st.caption("⚠️ For research use only. Not for clinical diagnosis.")
'''

_app_path = EXPORT_DIR / "dr_app.py"
_app_path.write_text(APP_CODE.strip())
print(f"✅ Streamlit app saved → {_app_path}")
print("\nTo run locally:")
print(f"  streamlit run {_app_path}")
print("\nTo deploy on Hugging Face Spaces:")
print("  1. Create a Space (type: Streamlit)")
print("  2. Upload dr_app.py + export/ directory")
print("  3. Add requirements.txt: torch timm albumentations opencv-python-headless streamlit")


## 📋 Step 23 — Final Summary

In [ ]:
state = st_load()
W = 68
print("="*W)
print("  DIABETIC RETINOPATHY GRADING — v19 PRODUCTION SUMMARY")
print("="*W)
print(f"  Backbone        : {CFG['model_name']}")
print(f"  Loss            : HybridLoss (0.5×WCE + 0.5×Focal) + label_smooth={CFG['label_smooth']}")
print(f"  Device          : {DEVICE} | AMP: {'ON' if USE_AMP else 'OFF'}")
print(f"  Dataset         : APTOS 2019 | Clean: {len(df):,} | Train: {len(df_trainval):,} | Test: {len(df_test):,}")
print()
print("  ─── K-Fold Cross-Validation ───")
if fold_best_qwks:
    for i,q in enumerate(fold_best_qwks):
        print(f"    Fold {i}: QWK = {q:.4f}")
    print(f"    Mean : {np.mean(fold_best_qwks):.4f} ± {np.std(fold_best_qwks):.4f}")
print(f"    OOF QWK (threshold-optimized): {state.get('oof_qwk','N/A')}")
print(f"    OOF Acc                      : {float(state.get('oof_acc',0))*100:.2f}%")
print()
print("  ─── Final Hold-Out Test ───")
print(f"    Test QWK : {state.get('test_qwk','N/A')}  "
      f"{'✅ TARGET MET' if float(state.get('test_qwk',0))>=0.90 else '⚠️  below 0.90'}")
print(f"    Test Acc : {float(state.get('test_acc',0))*100:.2f}%")
print()
print("  ─── Exported Artifacts ───")
for f in sorted(EXPORT_DIR.glob("*")):
    sz = f.stat().st_size
    print(f"    {f.name:<35s}  {sz/1e6:.2f} MB" if sz>1e5 else f"    {f.name}")
print()
print("  ─── Plots ───")
for p in sorted(PLOT_DIR.glob("*.png")):
    print(f"    {p.name}")
print("="*W)
print("  ⚠️  RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("="*W)
